gsri-18/ISEAR-dataset-complete : Fetches the International Survey on Emotion Antecedents and Reactions dataset.

In [ ]:
# --- SETUP & LIBRARIES ---
from google.colab import drive
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset, DatasetDict
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding)
from sklearn.metrics import classification_report, confusion_matrix, multilabel_confusion_matrix,f1_score, accuracy_score, precision_recall_fscore_support
import evaluate
import time

# Mount Drive
# drive.mount('/content/drive')
DRIVE_PATH = "/content/drive/MyDrive/Colab Notebooks/MoodLens"
OUTPUT_DIR = f"{DRIVE_PATH}/checkpoints"
FINAL_MODEL_DIR = f"{DRIVE_PATH}/final_moodlens_model"

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)


In [ ]:
# --- LOAD & SPLIT DATASET ---
# Load directly from Hugging Face
raw_ds =load_dataset("gsri-18/ISEAR-dataset-complete")

# Rename 'emotion' to 'label' for consistency
raw_ds = raw_ds.rename_column("emotion", "label")

# 80/10/10 Split
train_test = raw_ds['train'].train_test_split(test_size=0.2, seed=42)
test_valid = train_test['test'].train_test_split(test_size=0.5, seed=42)

dataset = DatasetDict({
    'train': train_test['train'],
    'valid': test_valid['train'],
    'test': test_valid['test']
})


In [ ]:
# --- TOKENIZATION & LABEL MAPPING ---
model_checkpoint = "distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

label_list = ['joy', 'sadness', 'guilt', 'shame', 'fear', 'anger', 'disgust']
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

def preprocess_function(examples):
    # Using 512 for max accuracy on 200-word entries
    tokenized = tokenizer(examples["content"], truncation=True, padding="max_length", max_length=512)

    # Multi-label formatting
    labels_matrix = []
    for label_str in examples["label"]:
        label_vector = [0.0] * 7
        if label_str in label2id:
            label_vector[label2id[label_str]] = 1.0
        labels_matrix.append(label_vector)

    tokenized["labels"] = labels_matrix
    return tokenized

tokenized_ds = dataset.map(preprocess_function, batched=True)

# Remove string columns so Trainer only sees Tensors
# We remove 'content' and 'label' (the strings) leaving 'input_ids', 'attention_mask', and 'labels'
cols_to_remove = dataset["train"].column_names
tokenized_ds = tokenized_ds.remove_columns(cols_to_remove)


In [ ]:
# --- MODEL SETUP ---
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    problem_type="multi_label_classification",
    id2label=id2label,
    label2id=label2id
)


In [ ]:
# --- METRICS & TRAINING ---
clf_metrics = evaluate.combine(["accuracy", "f1", "precision", "recall"])

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Apply sigmoid to get probabilities, then threshold at 0.5
    probs = torch.sigmoid(torch.Tensor(logits)).numpy()
    predictions = (probs > 0.5).astype(float)

    # Calculate weighted metrics
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="weighted")
    acc = accuracy_score(labels, predictions)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",      # Updated for Transformers 5.x
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=2,
    fp16=True,
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["valid"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# Start Training
print(" Training MoodLens...")
trainer.train(resume_from_checkpoint=True if os.path.exists(OUTPUT_DIR) and os.listdir(OUTPUT_DIR) else False)

# Save Final
model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print(f" Model saved to {FINAL_MODEL_DIR}")